In [ ]:
from google.colab import drive
import zipfile

drive.mount('/content/drive')
ZIP_FILE_PATH = '/content/drive/MyDrive/DS Project/Copy of bbdd_kidney.zip'
EXTRACT_DIR = '/content/dataset'
with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

In [ ]:
import os
import glob
import numpy as np
from PIL import Image, ImageFile
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

ImageFile.LOAD_TRUNCATED_IMAGES = True


class KidneyDataset(Dataset):
    def __init__(self, image_paths, labels, mean, std):
        self.image_paths = image_paths
        self.labels = labels
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert("RGB")
        img_tensor = TF.to_tensor(img)

        dtype, device = img_tensor.dtype, img_tensor.device
        mean_tensor = torch.as_tensor(self.mean, dtype=dtype, device=device)
        std_tensor = torch.as_tensor(self.std, dtype=dtype, device=device)
        norm_img = (img_tensor - mean_tensor[:, None, None]) / std_tensor[:, None, None]
        label = self.labels[idx]
        return {
            "image": norm_img,
            "label": torch.tensor(label, dtype=torch.long)
        }


def extract_all_labels(image_paths, labels_dir):
    labels = []
    for img_path in image_paths:
        file_stem = os.path.splitext(os.path.basename(img_path))[0]
        txt_path = os.path.join(labels_dir, f"{file_stem}.txt")
        if not os.path.exists(txt_path):
            raise FileNotFoundError(f"Missing label file for image '{os.path.basename(img_path)}'. Expected at: {txt_path}")
        with open(txt_path, 'r') as f:
            _ = f.readline()
            diag_line = f.readline()
            label_int = int(diag_line.split('Global diagnosis: ')[-1].split('\n')[0])
            labels.append(label_int)
    return labels


def pad_collate_fn(batch):
  # Our images do not all have the same dimensions, so if within our batch there are
  # smaller images, we pad them, so that they match the rest of the images.
    images = [item["image"] for item in batch]
    labels = [item["label"] for item in batch]

    max_h = max(img.shape[1] for img in images)
    max_w = max(img.shape[2] for img in images)

    padded_images = []
    for img in images:
        c, h, w = img.shape
        pad_h = max_h - h
        pad_w = max_w - w
        padded_img = F.pad(img, (0, pad_w, 0, pad_h), value=0.0)
        padded_images.append(padded_img)

    padded_images = torch.stack(padded_images, dim=0)
    labels = torch.tensor(labels, dtype=torch.long)

    return {"image": padded_images, "label": labels}


def compute_dataset_stats(train_paths):
    print("Computing mean and std for training set...")
    mean = torch.zeros(3)
    std = torch.zeros(3)

    for path in train_paths:
        img = Image.open(path).convert("RGB")
        tensor = TF.to_tensor(img)
        mean += tensor.mean(dim=(1, 2))
        std += tensor.std(dim=(1, 2))

    mean /= len(train_paths)
    std /= len(train_paths)

    print(f"Calculated Mean: {mean.tolist()}")
    print(f"Calculated Std:  {std.tolist()}")
    return mean.tolist(), std.tolist()


def build_efficientnet_model(num_classes=2):
    model = timm.create_model("efficientnet_b0", pretrained=True, in_chans=3)
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

def evaluate_auc(model, dataloader, device):
    model.eval()
    all_targets, all_probs = [], []

    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            all_targets.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    auc_score = roc_auc_score(all_targets, all_probs)
    return auc_score, np.array(all_targets), np.array(all_probs)


def run_pipeline():
    IMAGES_DIR = "dataset/images"
    LABELS_DIR = "dataset/labels"
    BATCH_SIZE = 16
    EPOCHS = 15
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    image_paths = sorted(glob.glob(os.path.join(IMAGES_DIR, "*.jpg")))

    all_labels = extract_all_labels(image_paths, LABELS_DIR)
    # 80% Train+Val, 20% Test Set
    train_val_files, test_files, train_val_labels, test_labels = train_test_split(
        image_paths,
        all_labels,
        test_size=0.20,
        stratify=all_labels,
        random_state=42)

    # 80% Train and 20% Val
    train_files, val_files, train_labels, val_labels = train_test_split(
        train_val_files,
        train_val_labels,
        test_size=0.20,
        stratify=train_val_labels,
        random_state=42)

    # Compute statistics on training set only to prevent data leakage
    mean, std = compute_dataset_stats(train_files)

    # Instantiate datasets with computed stats
    train_ds = KidneyDataset(train_files, train_labels, mean, std)
    val_ds   = KidneyDataset(val_files, val_labels, mean, std)
    test_ds  = KidneyDataset(test_files, test_labels, mean, std)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_collate_fn)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_collate_fn)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=pad_collate_fn)

    model = build_efficientnet_model(num_classes=2).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss()

    best_val_auc = 0.0

    print("Training EfficientNet-B0...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0

        for batch in train_loader:
            images = batch["image"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        val_auc, _, _ = evaluate_auc(model, val_loader, DEVICE)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), "best_efficientnet.pth")

        print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] - Loss: {running_loss/len(train_ds):.4f} | Val ROC-AUC: {val_auc:.4f}")

    # Evaluate Holdout Test Set
    print("\nEvaluating on Unseen Test Set...")
    model.load_state_dict(torch.load("best_efficientnet.pth"))

    test_auc, y_true, y_probs = evaluate_auc(model, test_loader, DEVICE)
    print(f"FINAL HOLDOUT TEST ROC-AUC: {test_auc:.4f}")

    # Plot ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Test ROC Curve (AUC = {test_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Test Set ROC Curve')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.savefig("test_roc_curve.png", dpi=300)
    plt.close()

if __name__ == "__main__":
    run_pipeline()